# Fine-Tuning a Mixture-of-Experts Chatbot on an RTX 4050

This notebook fine-tunes a small **Mixture-of-Experts (MoE)** language model into a helpful chatbot, using techniques that fit inside **8–16 GB of VRAM** (RTX 4050 mobile/desktop).

**Model choices (swap in Section 2):**
- `Qwen/Qwen1.5-MoE-A2.7B-Chat` — 14.3B total / 2.7B active params, 60 routing + 4 shared experts.
- `microsoft/Phi-mini-MoE-instruct` — 7.6B total / 2.4B active params, distilled from Phi-3.5-MoE (recommended default — smaller footprint).

**Techniques used:**
1. **4-bit NF4 quantization** via `bitsandbytes` (with double quantization + bf16 compute).
2. **LoRA** adapters targeting attention *and* MoE expert / gate modules via `peft`.
3. **Gradient checkpointing** to trade compute for memory.
4. **Paged 8-bit AdamW** optimizer to avoid OOM on the optimizer state.
5. **MoE auxiliary load-balancing loss** (`router_aux_loss_coef`, `output_router_logits`) so experts don't collapse.
6. **Chat template** formatting + assistant-only loss masking.
7. **Gradio** `ChatInterface` with streaming for the live demo.

> Run cells top-to-bottom. The training section will work on ~2k Alpaca samples in roughly 30–90 minutes on an RTX 4050. Tune `MAX_SAMPLES`, `MAX_SEQ_LEN`, and `NUM_EPOCHS` for your budget.


## 1. Install dependencies

Install once per environment. Torch should be installed separately with the correct CUDA build for your system — see [pytorch.org](https://pytorch.org/get-started/locally/).


In [1]:
# Uncomment and run once. Keep your PyTorch build aligned with your CUDA version.
# %pip install -q --upgrade "transformers>=4.45.0" "accelerate>=0.33" "peft>=0.12" \
#     "bitsandbytes>=0.43" "datasets>=2.20" "trl>=0.11" "gradio>=4.44" \
#     "sentencepiece" "einops" "scipy" "evaluate"


In [2]:
!pip install -q --upgrade "transformers>=4.45.0" "accelerate>=0.33" "peft>=0.12

## 2. Environment & GPU check

Confirms CUDA is visible and reports VRAM. If you see less than ~7 GB free, reduce `MAX_SEQ_LEN` or switch to `Phi-tiny-MoE-instruct`.


In [3]:
import os, gc, math, json, random, time
import torch

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
# Helps PyTorch allocator avoid fragmentation during training.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

SEED = 42
random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

def show_gpu():
    if not torch.cuda.is_available():
        print("CUDA not available — training will not run on CPU.")
        return
    props = torch.cuda.get_device_properties(0)
    free, total = torch.cuda.mem_get_info(0)
    print(f"GPU: {props.name}")
    print(f"Total VRAM : {total/1e9:5.2f} GB")
    print(f"Free VRAM  : {free/1e9:5.2f} GB")
    print(f"Compute cap: {props.major}.{props.minor}")
    print(f"bf16 ok?   : {torch.cuda.is_bf16_supported()}")

show_gpu()


GPU: NVIDIA GeForce RTX 4050 Laptop GPU
Total VRAM :  6.44 GB
Free VRAM  :  5.32 GB
Compute cap: 8.9
bf16 ok?   : True


## 3. Configuration

All knobs live in one place. The defaults target an RTX 4050 (8 GB). If you have 12–16 GB, raise `MAX_SEQ_LEN` to 1024 or `PER_DEVICE_BATCH` to 2.


In [4]:
from dataclasses import dataclass, field
from typing import List

@dataclass
class CFG:
    # --- Model ---
    # Pick ONE. Phi-mini-MoE is the smallest/fastest option.
    model_id: str = "microsoft/Phi-mini-MoE-instruct"
    # model_id: str = "Qwen/Qwen1.5-MoE-A2.7B-Chat"

    # --- Data ---
    dataset_id: str = "tatsu-lab/alpaca"  # small, permissive, instruction-tuned
    max_samples: int = 2000               # bump for higher quality
    eval_split: float = 0.02              # 2% held-out
    max_seq_len: int = 768                # 512–1024 on 8GB, 2048 on 16GB

    # --- LoRA ---
    lora_r: int = 8
    lora_alpha: int = 16
    lora_dropout: float = 0.05

    # --- Training ---
    num_epochs: float = 1.0
    per_device_batch: int = 1
    grad_accum_steps: int = 16            # effective batch = 16
    learning_rate: float = 2e-4
    warmup_ratio: float = 0.03
    weight_decay: float = 0.0
    # MoE load-balancing coefficient. 0.001–0.01 is typical.
    router_aux_loss_coef: float = 0.01

    # --- Output ---
    output_dir: str = "./moe-chatbot-lora"
    logging_steps: int = 10
    save_steps: int = 200
    eval_steps: int = 200

cfg = CFG()
print(json.dumps(cfg.__dict__, indent=2))


{
  "model_id": "microsoft/Phi-mini-MoE-instruct",
  "dataset_id": "tatsu-lab/alpaca",
  "max_samples": 2000,
  "eval_split": 0.02,
  "max_seq_len": 768,
  "lora_r": 8,
  "lora_alpha": 16,
  "lora_dropout": 0.05,
  "num_epochs": 1.0,
  "per_device_batch": 1,
  "grad_accum_steps": 16,
  "learning_rate": 0.0002,
  "warmup_ratio": 0.03,
  "weight_decay": 0.0,
  "router_aux_loss_coef": 0.01,
  "output_dir": "./moe-chatbot-lora",
  "logging_steps": 10,
  "save_steps": 200,
  "eval_steps": 200
}


## 4. Load the model in 4-bit + enable MoE router outputs

Key bits:
- `BitsAndBytesConfig` uses **NF4** (4-bit NormalFloat) with **double quantization** and **bf16** compute — the sweet spot for consumer GPUs.
- We pass `output_router_logits=True` and set `router_aux_loss_coef` **before** loading so the forward pass emits router logits and the model's built-in auxiliary loss is added to `outputs.loss` during training. This is the MoE **load-balancing loss** that keeps experts from collapsing onto a few popular ones.
- `prepare_model_for_kbit_training` casts layernorms to fp32, enables input grads, and turns on gradient checkpointing compatibly with 4-bit weights.


In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, AutoConfig
from peft import prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(cfg.model_id, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
# Right-padding is correct for causal LM training.
tokenizer.padding_side = "right"

# Tell the config to emit router logits so the aux load-balancing loss is computed.
model_config = AutoConfig.from_pretrained(cfg.model_id, trust_remote_code=True)
if hasattr(model_config, "output_router_logits"):
    model_config.output_router_logits = True
if hasattr(model_config, "router_aux_loss_coef"):
    model_config.router_aux_loss_coef = cfg.router_aux_loss_coef

model = AutoModelForCausalLM.from_pretrained(
    cfg.model_id,
    config=model_config,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    attn_implementation="eager",  # safer with grad-checkpointing on Ampere+ consumer GPUs
)

# Sanity prints: confirm MoE aux loss is wired up.
print("output_router_logits :", getattr(model.config, "output_router_logits", None))
print("router_aux_loss_coef :", getattr(model.config, "router_aux_loss_coef", None))
print("num_experts (if any) :", getattr(model.config, "num_experts",
                                       getattr(model.config, "num_local_experts", None)))

model = prepare_model_for_kbit_training(
    model, use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)
# Generation caching must be off while training with checkpointing.
model.config.use_cache = False


c:\Users\saad7\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\saad7\AppData\Local\Programs\Python\Python310\lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\saad7\.cache\huggingface\hub\models--microsoft--Phi-mini-MoE-instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Pytho

ImportError: This modeling file requires the following packages that were not found in your environment: einops, flash_attn. Run `pip install einops flash_attn`

## 5. Attach LoRA adapters (attention + MoE experts)

MoE models have two groups of trainable-ish modules worth adapting:
1. **Attention projections** — `q_proj`, `k_proj`, `v_proj`, `o_proj` (always).
2. **Expert / gate modules** — names vary by architecture:
   - Qwen1.5-MoE: `gate`, `shared_expert_gate`, `shared_expert.{gate,up,down}_proj`, `experts.{i}.{gate,up,down}_proj`
   - Phi-mini-MoE: `gate`, `w1`, `w2`, `w3` (expert MLPs)

Rather than hard-coding every expert index, we regex-match with `target_modules` and let PEFT expand it.


In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

# Auto-pick target modules based on the architecture. Add more if you inspect model.named_modules().
if "qwen" in cfg.model_id.lower():
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ]
elif "phi" in cfg.model_id.lower():
    target_modules = [
        "qkv_proj", "o_proj",            # phi packs qkv
        "gate_up_proj", "down_proj",      # expert MLP layout in phi-moe
        "q_proj", "k_proj", "v_proj",     # fallback if a variant splits them
        "w1", "w2", "w3",                 # mixtral-style expert names (fallback)
    ]
else:
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"]

lora_config = LoraConfig(
    r=cfg.lora_r,
    lora_alpha=cfg.lora_alpha,
    lora_dropout=cfg.lora_dropout,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=target_modules,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


## 6. Load and format the instruction dataset

We use a **small slice** of Alpaca (~2k rows) so training completes in tens of minutes. Each row becomes a single-turn chat:

```
<system> You are a helpful assistant. </system>
<user> {instruction + optional input} </user>
<assistant> {output} </assistant>
```

Then the model's **chat template** (`tokenizer.apply_chat_template`) renders the correct special tokens for whichever MoE you picked.


In [ ]:
from datasets import load_dataset

raw = load_dataset(cfg.dataset_id, split="train")
raw = raw.shuffle(seed=SEED).select(range(min(cfg.max_samples, len(raw))))

SYSTEM_PROMPT = "You are a helpful, concise, and friendly AI assistant."

def to_messages(example):
    user = example["instruction"].strip()
    if example.get("input"):
        user += "\n\n" + example["input"].strip()
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user},
            {"role": "assistant", "content": example["output"].strip()},
        ]
    }

ds = raw.map(to_messages, remove_columns=raw.column_names)
split = ds.train_test_split(test_size=cfg.eval_split, seed=SEED)
train_ds, eval_ds = split["train"], split["test"]
print(f"train: {len(train_ds)}  eval: {len(eval_ds)}")
print("Example:\n", json.dumps(train_ds[0]["messages"], indent=2)[:600])


### 6a. Tokenize with assistant-only loss masking

We tokenize the full rendered chat, but **mask everything before the assistant turn to `-100`** so the model only computes loss on its own response — this is standard SFT practice and dramatically improves quality vs. training on the whole sequence.


In [ ]:
IGNORE_INDEX = -100

def tokenize(example):
    msgs = example["messages"]
    # Full sequence with the assistant turn included (for labels + input_ids).
    full_ids = tokenizer.apply_chat_template(
        msgs, tokenize=True, add_generation_prompt=False,
        truncation=True, max_length=cfg.max_seq_len,
    )
    # Prompt portion only (system + user) — so we know where assistant starts.
    prompt_ids = tokenizer.apply_chat_template(
        msgs[:-1], tokenize=True, add_generation_prompt=True,
        truncation=True, max_length=cfg.max_seq_len,
    )
    labels = list(full_ids)
    cut = min(len(prompt_ids), len(labels))
    for i in range(cut):
        labels[i] = IGNORE_INDEX
    return {"input_ids": full_ids, "labels": labels, "attention_mask": [1]*len(full_ids)}

train_tok = train_ds.map(tokenize, remove_columns=train_ds.column_names, desc="tokenize-train")
eval_tok  = eval_ds.map(tokenize,  remove_columns=eval_ds.column_names,  desc="tokenize-eval")

# Peek at label masking.
sample = train_tok[0]
n_masked = sum(1 for x in sample["labels"] if x == IGNORE_INDEX)
print(f"seq len={len(sample['input_ids'])}  masked(prompt)={n_masked}  learn(assistant)={len(sample['labels'])-n_masked}")


### 6b. Dynamic padding data collator

Pads each batch to the longest example in the batch (not the global max) — saves a lot of VRAM and time.


In [ ]:
from dataclasses import dataclass
from typing import Dict, List

@dataclass
class PadCollator:
    pad_id: int
    label_pad_id: int = IGNORE_INDEX

    def __call__(self, features: List[Dict]) -> Dict[str, torch.Tensor]:
        maxlen = max(len(f["input_ids"]) for f in features)
        batch = {"input_ids": [], "attention_mask": [], "labels": []}
        for f in features:
            pad = maxlen - len(f["input_ids"])
            batch["input_ids"].append(f["input_ids"] + [self.pad_id]*pad)
            batch["attention_mask"].append(f["attention_mask"] + [0]*pad)
            batch["labels"].append(f["labels"] + [self.label_pad_id]*pad)
        return {k: torch.tensor(v, dtype=torch.long) for k, v in batch.items()}

collator = PadCollator(pad_id=tokenizer.pad_token_id)


## 7. Training

We use Hugging Face `Trainer`. Key VRAM-saving flags:

| Flag | Effect |
|------|--------|
| `bf16=True` | Mixed precision (fall back to `fp16` on older GPUs). |
| `gradient_checkpointing=True` | Recompute activations — ~30–40% VRAM saved. |
| `optim="paged_adamw_8bit"` | 8-bit optimizer states w/ paged offload. |
| `per_device_train_batch_size=1` + `gradient_accumulation_steps=16` | Effective batch 16 without the memory cost. |
| `group_by_length=True` | Pads less within each batch. |

The MoE auxiliary load-balancing loss is **automatically added to `outputs.loss`** by the model because we set `output_router_logits=True` and `router_aux_loss_coef` on the config. You'll see the combined loss in the logs.


In [ ]:
from transformers import Trainer, TrainingArguments

use_bf16 = torch.cuda.is_bf16_supported()

args = TrainingArguments(
    output_dir=cfg.output_dir,
    num_train_epochs=cfg.num_epochs,
    per_device_train_batch_size=cfg.per_device_batch,
    per_device_eval_batch_size=cfg.per_device_batch,
    gradient_accumulation_steps=cfg.grad_accum_steps,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    learning_rate=cfg.learning_rate,
    lr_scheduler_type="cosine",
    warmup_ratio=cfg.warmup_ratio,
    weight_decay=cfg.weight_decay,
    optim="paged_adamw_8bit",
    bf16=use_bf16,
    fp16=not use_bf16,
    logging_steps=cfg.logging_steps,
    save_steps=cfg.save_steps,
    save_total_limit=2,
    eval_strategy="steps",
    eval_steps=cfg.eval_steps,
    group_by_length=True,
    report_to="none",
    seed=SEED,
    # Keep this off during training; we'll flip it back on for inference.
    # Helps when the dataloader tries to pin fewer workers on Windows/WSL.
    dataloader_num_workers=2,
    remove_unused_columns=False,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_tok,
    eval_dataset=eval_tok,
    data_collator=collator,
    tokenizer=tokenizer,
)

# Free any stale cache before launching.
gc.collect(); torch.cuda.empty_cache() if torch.cuda.is_available() else None


### 7a. Kick off training

If you OOM, the simplest fixes in order are: **lower `cfg.max_seq_len` → 512**, **raise `grad_accum_steps`**, **switch to `Phi-tiny-MoE-instruct`**, or **reduce `cfg.max_samples`**.


In [ ]:
train_result = trainer.train()
trainer.save_model(cfg.output_dir)         # saves LoRA adapters only
tokenizer.save_pretrained(cfg.output_dir)

metrics = train_result.metrics
metrics["train_samples"] = len(train_tok)
trainer.log_metrics("train", metrics)
trainer.save_metrics("train", metrics)


## 8. Evaluation

We run two complementary evals:

1. **Held-out perplexity** — objective, cheap, correlates with fluency.
2. **Qualitative MT-Bench-style prompts** — spot-check helpfulness, reasoning, and coding across canonical categories.


In [ ]:
import math as _math

eval_metrics = trainer.evaluate()
if "eval_loss" in eval_metrics:
    eval_metrics["eval_perplexity"] = _math.exp(min(20, eval_metrics["eval_loss"]))
trainer.log_metrics("eval", eval_metrics)
trainer.save_metrics("eval", eval_metrics)
eval_metrics


### 8a. MT-Bench-style qualitative probes

Small hand-picked set covering **writing / reasoning / math / coding / extraction / roleplay / knowledge / safety**. Scoring MT-Bench properly needs an LLM judge (GPT-4 etc.) — here we just generate samples so you can eyeball quality. Plug in an external judge API if you want numeric scores.


In [ ]:
# Re-enable cache for fast generation.
model.config.use_cache = True
model.eval()

MT_BENCH_LITE = [
    ("writing",   "Write a 4-sentence cheerful birthday message for my grandmother."),
    ("reasoning", "If a train leaves Riyadh at 8:00 AM going 120 km/h and another leaves Jeddah (950 km away) at 9:00 AM going 100 km/h toward Riyadh, at what time do they meet?"),
    ("math",      "Solve for x: 3(x - 4) + 2 = 2x + 7. Show each step."),
    ("coding",    "Write a Python function `is_palindrome(s)` that ignores punctuation and case. Include 3 test cases."),
    ("extraction","From the sentence 'Dr. Sara Al-Otaibi joined Acme Corp in March 2024 as CTO', extract name, title, company, and start date as JSON."),
    ("roleplay",  "You are a calm meditation coach. Lead me through a 30-second breathing exercise."),
    ("knowledge", "In 3 bullets, explain what a Mixture-of-Experts model is and why it's cheaper to serve than a dense model of the same capacity."),
    ("safety",    "My friend feels very stressed about exams. What can I say to support them?"),
]

@torch.no_grad()
def chat(user_msg, system=SYSTEM_PROMPT, max_new_tokens=256, temperature=0.7, top_p=0.9):
    messages = [{"role":"system","content":system}, {"role":"user","content":user_msg}]
    input_ids = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)
    out = model.generate(
        input_ids,
        max_new_tokens=max_new_tokens,
        do_sample=temperature > 0,
        temperature=temperature,
        top_p=top_p,
        pad_token_id=tokenizer.pad_token_id,
    )
    return tokenizer.decode(out[0, input_ids.shape[1]:], skip_special_tokens=True).strip()

results = []
for cat, prompt in MT_BENCH_LITE:
    ans = chat(prompt, max_new_tokens=220)
    results.append({"category": cat, "prompt": prompt, "answer": ans})
    print(f"=== [{cat}] ===\nQ: {prompt}\nA: {ans}\n")

# Save for later review.
Path(cfg.output_dir).mkdir(exist_ok=True, parents=True)
with open(Path(cfg.output_dir)/"mt_bench_lite.json", "w") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)


### 8b. (Optional) LLM-as-a-judge scoring stub

If you have an API key for a judge model, rate each answer 1–10 on helpfulness. Left as a stub so this notebook has no external dependencies.

```python
# import openai
# client = openai.OpenAI()
# def judge(q, a):
#     rubric = "Rate 1-10 for helpfulness, correctness, clarity. Reply JSON."
#     r = client.chat.completions.create(model="gpt-4o-mini",
#         messages=[{"role":"system","content":rubric},
#                   {"role":"user","content":f"Q: {q}\nA: {a}"}])
#     return r.choices[0].message.content
```


## 9. Save, reload, and optionally merge adapters

The LoRA adapter is tiny (~tens of MB). For easier deployment you can **merge** it into the quantized base weights — but merging into 4-bit weights is lossy; for best quality reload in bf16 before merging.


In [ ]:
# Reload from disk as a sanity check.
from peft import PeftModel

print("Adapter saved to:", cfg.output_dir)
print("Size on disk:")
for p in Path(cfg.output_dir).rglob("*.safetensors"):
    print(f"  {p.name:40s} {p.stat().st_size/1e6:7.1f} MB")

# Optional merge (uncomment if you want a single standalone model).
# base = AutoModelForCausalLM.from_pretrained(cfg.model_id, torch_dtype=torch.bfloat16,
#                                             device_map="cpu", trust_remote_code=True)
# merged = PeftModel.from_pretrained(base, cfg.output_dir).merge_and_unload()
# merged.save_pretrained(cfg.output_dir + "-merged", safe_serialization=True)
# tokenizer.save_pretrained(cfg.output_dir + "-merged")


## 10. Gradio chat demo

Streaming chat interface with adjustable temperature, top-p, max tokens, and system prompt. Conversation history is preserved across turns.


In [ ]:
import gradio as gr
from threading import Thread
from transformers import TextIteratorStreamer

model.config.use_cache = True
model.eval()

def build_inputs(history, user_msg, system):
    messages = [{"role":"system","content":system}]
    for u, a in history:
        messages.append({"role":"user","content":u})
        if a:
            messages.append({"role":"assistant","content":a})
    messages.append({"role":"user","content":user_msg})
    return tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt"
    ).to(model.device)

def respond(user_msg, history, system, max_new_tokens, temperature, top_p):
    input_ids = build_inputs(history or [], user_msg, system)
    streamer = TextIteratorStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)
    gen_kwargs = dict(
        input_ids=input_ids,
        max_new_tokens=int(max_new_tokens),
        do_sample=temperature > 0,
        temperature=float(temperature),
        top_p=float(top_p),
        pad_token_id=tokenizer.pad_token_id,
        streamer=streamer,
    )
    Thread(target=model.generate, kwargs=gen_kwargs, daemon=True).start()
    partial = ""
    for token in streamer:
        partial += token
        yield partial

with gr.Blocks(title="MoE Chatbot (LoRA fine-tuned)") as demo:
    gr.Markdown("# MoE Chatbot\nFine-tuned with LoRA on a small instruction set.")
    with gr.Row():
        with gr.Column(scale=3):
            chat = gr.Chatbot(height=480)
            msg  = gr.Textbox(placeholder="Ask me anything…", label="Your message")
            with gr.Row():
                send = gr.Button("Send", variant="primary")
                clr  = gr.Button("Clear")
        with gr.Column(scale=1):
            sys_box = gr.Textbox(value=SYSTEM_PROMPT, label="System prompt", lines=4)
            max_tok = gr.Slider(32, 1024, value=256, step=16, label="Max new tokens")
            temp    = gr.Slider(0.0, 1.5, value=0.7, step=0.05, label="Temperature")
            topp    = gr.Slider(0.1, 1.0, value=0.9, step=0.05, label="Top-p")

    def user_submit(user_msg, history):
        return "", (history or []) + [[user_msg, None]]

    def bot_stream(history, system, mx, t, p):
        user_msg = history[-1][0]
        history[-1][1] = ""
        for partial in respond(user_msg, history[:-1], system, mx, t, p):
            history[-1][1] = partial
            yield history

    send.click(user_submit, [msg, chat], [msg, chat], queue=False).then(
        bot_stream, [chat, sys_box, max_tok, temp, topp], chat
    )
    msg.submit(user_submit, [msg, chat], [msg, chat], queue=False).then(
        bot_stream, [chat, sys_box, max_tok, temp, topp], chat
    )
    clr.click(lambda: None, None, chat, queue=False)

# Launch. In Jupyter, inline=True embeds the UI. share=True gives a public URL.
demo.queue().launch(share=False, inline=True, debug=False)


## 11. Troubleshooting cheat sheet

| Symptom | Fix |
|---|---|
| `CUDA out of memory` on load | Check that no other GPU app is running. Try `Phi-tiny-MoE-instruct`. Make sure `bnb_4bit_use_double_quant=True`. |
| OOM during training step | Lower `cfg.max_seq_len` → 512 or 384. Raise `grad_accum_steps` (keeps effective batch). |
| OOM during `generate()` | Set `max_new_tokens` lower, or disable sampling. Turn on `use_cache=True` only at inference. |
| `use_reentrant` warning / grads `None` | We already pass `use_reentrant: False`. Make sure PEFT ≥ 0.12 and transformers ≥ 4.45. |
| Loss is NaN immediately | Disable `bf16` and use `fp16=True`; or lower LR to `1e-4`. |
| MoE loss not decreasing / all traffic one expert | Raise `router_aux_loss_coef` to `0.02`, or increase dataset diversity. |
| `target_modules` not found | Print `[n for n,_ in model.named_modules()]` and update the list in Section 5. |
| Windows + bitsandbytes issues | Install the prebuilt wheel: `pip install bitsandbytes --index-url https://jllllll.github.io/bitsandbytes-windows-webui/`. |
| Gradio UI doesn't render inline | Call `demo.launch(share=True)` and open the URL in your browser. |

---

### Where to go next

- **Larger dataset.** Swap `tatsu-lab/alpaca` for `HuggingFaceH4/ultrachat_200k` or `teknium/OpenHermes-2.5` (sample ~10–50k rows).
- **Multi-turn.** Extend `to_messages` to accept multi-turn conversations; the collator and trainer already handle them.
- **DPO / ORPO.** After SFT, run a preference-tuning pass with `trl.DPOTrainer` on a small pairwise set for a big quality jump.
- **Quantized serving.** Export with `llama.cpp` or `vLLM` (MoE support) for production inference.
